In [1]:
import json 

In [11]:
if (2 < 7):
  semester = 2
else:
  semester = 1
print(semester) 

2


In [2]:
from tensorflow import keras
from tensorflow.keras.layers import Layer
from tensorflow.keras.saving import register_keras_serializable
import numpy as np
import pandas as pd


@register_keras_serializable(package="Custom")
class SliceLayer(Layer):
    def __init__(self, index, **kwargs):
        super().__init__(**kwargs)
        self.index = index

    def call(self, inputs):
        return inputs[:, self.index]

    def get_config(self):
        config = super().get_config()
        config.update({"index": self.index})
        return config
      
model = keras.models.load_model(
  "model/cf_model_v4.keras",
  custom_objects={"SliceLayer": SliceLayer})
      
df = pd.read_csv("model/data_preprocessed.csv")
# df = pd.read_csv("dataset/nilai_mahasiswa - prep.csv")

c:\repo\ta-be\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [3]:
# user_id <-> nim
nim_to_user = (
    df[["nim", "user"]]
    .drop_duplicates()
    .set_index("nim")["user"]
    .to_dict()
)

user_to_nim = {v: k for k, v in nim_to_user.items()}

# item_id <-> nama matkul
item_to_matkul = (
    df[["item", "matkul"]]
    .drop_duplicates()
    .set_index("item")["matkul"]
    .to_dict()
)

all_items = df["item"].unique()

In [4]:
# save item_to_matkul
with open("model/item_to_matkul.json", "w") as f:
    f.write(str(item_to_matkul))

In [5]:
# load item_to_matkul
with open("model/item_to_matkul.json", "r") as f:
    item_to_matkul_load = eval(f.read())

In [6]:
nim_to_user["H1101221016"]

1160

In [7]:
def predict_pair(user_id, item_id):
    x = np.array([[user_id, item_id]], dtype=np.float32)
    return float(model.predict(x, verbose=0)[0][0])

In [8]:
def recommend_by_nim(nim, top_k=5):
    if nim not in nim_to_user:
        raise ValueError(f"NIM {nim} tidak ditemukan di data training")

    user_id = nim_to_user[nim]

    preds = []
    selecter_item = [86,87]
    for item_id in selecter_item:
        score = predict_pair(user_id, item_id)
        preds.append((item_id, score))

    # Urutkan skor tertinggi
    preds = sorted(preds, key=lambda x: x[1], reverse=True)[:top_k]

    # Format hasil
    result = []
    for item_id, score in preds:
        result.append({
            "item": int(item_id),
            "nama_matkul": item_to_matkul[item_id],
            "score": round(score, 4)
        })

    return result

In [9]:
nim_input = "H1101221016"
rekomendasi = recommend_by_nim(nim_input, 10)

for i, r in enumerate(rekomendasi, 1):
    print(f"{i}. {r['nama_matkul']} (item={r['item']} | score={r['score']})")

1. Manajemen Produk (item=86 | score=0.9707)
2. Manajemen Proses Bisnis (item=87 | score=0.9637)
